# Nemotron Nano Omni V3 GRPO Training via NeMo Gym

## Overview
This notebook trains **Nemotron Nano V3 Omni** on the **MMPR-Tiny** visual reasoning dataset using [NemoRL](https://github.com/NVIDIA-NeMo/RL) as the training engine while [NeMo Gym](https://github.com/NVIDIA-NeMo/Gym) handles rollout. 

NeMo Gym lets you easily swap in custom environments, verifiers, and agents during RL training without needing to modify NeMo RL's code.

The goal is to improve the model's ability to
- parse and understand a math diagram or chart from an image,
- reason step-by-step inside a `<think>` block, and
- produce the final answer in `\\boxed{...}` format.

### Dataset: OpenGVLab/MMPR-Tiny

[MMPR-Tiny](https://huggingface.co/datasets/OpenGVLab/MMPR-Tiny) is a compact public subset of the **MMPR** (Multimodal Math Process Reward) dataset family, containing image–question–answer triples sourced from mathematical reasoning benchmarks.

**Notes**:
- All commands in this notebook are intended to be copied and run in the relevant interactive shell on the head node or worker nodes.
- In production, submit as a Slurm batch job. The interactive path here is for iteration and debugging.

## Prerequisites

- **Compute**: 1× DGX H100 node (8× H100 80 GB) or equivalent on a Slurm cluster.

- **Storage**: High-speed shared network filesystem accessible from all nodes. We assume it is at `</YOUR/SHARED/NETWORK/STORAGE>` on the host, mounted as `/shared` inside the container.

We assume the code, models, checkpoint and data assets are stored under the following directory structure. 

```
</YOUR/SHARED/NETWORK/STORAGE> (host) : /shared (container)
├── code/
│   ├── RL/                          # NeMo RL root directory
│   └── Nemotron/usage-cookbook/Nemotron-3-Nano-Omni/grpo_nemo_gym      # this notebook lives here
├── models/
│   └── Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16/
├── checkpoints/                     # trained model checkpoints (~425 GB each)
├── data/
│   ├── mmpr_tiny/                   # MMPR-Tiny parquet + images (HF cache, populated by converter)
│   └── mmpr_tiny_gym/               # NeMo Gym JSONL files (written by converter)
└── HF_HOME/                         # HuggingFace cache directory
```

- **Model**: Nemotron Omni-V3 checkpoint downloaded from HuggingFace: [https://huggingface.co/nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16](https://huggingface.co/nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16).

In [ ]:
mkdir -p </YOUR/SHARED/NETWORK/STORAGE>/models
cd </YOUR/SHARED/NETWORK/STORAGE>/models
export HF_TOKEN=<YOUR_HF_TOKEN>

hf download nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16 \
  --local-dir Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16

# Or from NGC:
# ngc registry model download-version "nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16" 

- **Docker image**: Build the NeMo RL container with NeMo Gym support.

Clone the repo and check out the correct branch:

In [ ]:
mkdir -p </YOUR/SHARED/NETWORK/STORAGE>/code
cd </YOUR/SHARED/NETWORK/STORAGE>/code

git clone --recursive -b nano-v3-omni https://github.com/NVIDIA-NeMo/RL

Build the Docker image and push to a registry accessible to all nodes (e.g., NGC or Docker Hub):

In [ ]:
cd </YOUR/SHARED/NETWORK/STORAGE>/code/RL
rm -rf .lock-venv

TAG="$(git rev-parse --short HEAD)"

docker buildx build \
  --target release \
  --build-arg MAX_JOBS=4 \
  --build-arg VLLM_PRECOMPILED_WHEEL_LOCATION="https://wheels.vllm.ai/27ca95b3c9e6e32ac02508e729c01865800fd036/vllm-0.14.0rc2.dev196%2Bg27ca95b3c-cp38-abi3-manylinux_2_31_x86_64.whl" \
  --build-arg SETUPTOOLS_SCM_PRETEND_VERSION_FOR_VLLM="0.14.0rc2.dev196+g27ca95b3c" \
  -f docker/Dockerfile \
  --tag "nemo-rl:nano-v3-vl-${TAG}" \
  --load .

## Step 1. Prepare the MMPR-Tiny dataset in NeMo Gym JSONL format

NeMo Gym standardizes training data on the OpenAI Responses API schema where each JSONL line is a responses_create_params object with the input prompt and any verifier metadata needed for reward computation.

This conversion step need to be run **once** before the first training. It downloads MMPR-Tiny from HuggingFace if not cached, then writes the filtered JSONL files for the **MCQ + numeric subset** (~71% of the full dataset, ~45 000 train / ~468 val rows).

MMPR-Tiny contains three question types after conversion:

| Type | Share | Agent | Grading | LLM judge? |
|---|---|---|---|---|
| Multiple-choice (single-letter A–G + visible options) | ~33% | `mcqa_simple_agent` | Exact-match on `\boxed{X}` | Never |
| Numeric (integer, decimal, simple fraction) | ~38% | `math_with_judge_simple_agent` | `math_verify` symbolic library | Never (`should_use_judge: false`) |
| Free-form (expressions, open text) | ~29% | `equivalence_llm_judge_simple_agent` | Requires LLM judge | Always |

We use `--question-type mcqa_and_numeric` to keep both rule-based subsets and exclude the free-form samples that would require an LLM judge. This covers ~71% of the dataset with fully deterministic grading.

Key implementation details of the converter:
- `mcqa` samples: single-letter answer (`A`–`G`) **and** visible letter choices detected in the question body → routed to `mcqa_simple_agent`.
- `numeric` samples: answer matches `[+-]?\d+(\.\d+)?(/\d+)?` (integers, decimals, simple fractions) → routed to `math_with_judge_simple_agent`.
- Image file paths are stored as local paths; `nemo_rl/environments/nemo_gym.py:encode_images_in_examples()` converts them to base64 data-URIs automatically at training time.

For this step, start an interactive NemoRL job (modest resource shall be sufficient, either a CPU only node or a single GPU slurm job).

In [ ]:
# Run from the root of NeMo RL repo
NUM_ACTOR_NODES=1  # Total nodes requested (head is colocated on ray-worker-0)

export CONTAINER="nemo-rl:nano-v3-vl-<SUPPLY_YOUR_TAG>"     # supply the correct container image tag
export MOUNTS="/lustre:/lustre,</YOUR/SHARED/NETWORK/STORAGE>:/shared"

sbatch \
    --nodes=${NUM_ACTOR_NODES} \
    --account=<YOUR_SLURM_ACCOUNT> \
    --job-name=nano_omni \
    --partition=<YOUR_SLURM_PARTITION> \
    --time=4:0:0 \
    --gres=gpu:1 \
    ray.sub


Upon successful submission, Slurm will print the SLURM_JOB_ID, for example:
```
Submitted batch job 1980204
```

Once the Ray cluster is up, a script will be created to attach to the Ray head node. Run this script to launch experiments:
```
bash 1980204-attach.sh
``` 

See https://github.com/NVIDIA-NeMo/RL/blob/main/docs/cluster.md for further information on interactive and batch cluster setup. 


Now that you are on the Ray head node, you can launch the training command as follows:


In [ ]:
# Run from the NeMo RL root directory (inside the container or in a CPU-only pre-process job)
CACHE=/shared/data/mmpr_tiny              # parquet + images downloaded here
OUT=/shared/data/mmpr_tiny_gym

python /shared/code/Nemotron/usage-cookbook/Nemotron-3-Nano-Omni/grpo_nemo_gym/convert_mmpr_tiny_to_gym_jsonl.py \
    --cache-dir ${CACHE} \
    --output-dir ${OUT} \
    --question-type mcqa_and_numeric      # MCQ (exact-match) + numeric (math_verify), no LLM judge

# Expected output:
#   Total samples: N → train: N-500, val: 500
#   Question type filter: 'mcqa_and_numeric'
#   Converting train split ...
#     Wrote ~45000 rows (0 skipped, ~18000 filtered) → .../mmpr_tiny_train_mcqa_and_numeric.jsonl
#       mcqa_simple_agent: 20924
#       math_with_judge_simple_agent: ~24000
#   Converting val split ...
#     Wrote ~468 rows (0 skipped, ~32 filtered) → .../mmpr_tiny_val_mcqa_and_numeric.jsonl
#       mcqa_simple_agent: 168
#       math_with_judge_simple_agent: ~300

## Step 2. Review the training config

The config file `grpo_mmpr_tiny_nanov3omni_gym.yaml` (in the same folder as this notebook) is tuned for **1× DGX H100** (8 GPUs total) with the NeMo Gym rollout path.

### Key NeMo Gym–specific settings

```yaml
grpo:
  deduplicate_multimodal_data: false   # MUST be false — incompatible with async_engine
  max_val_samples: null                # null = use all val samples

policy:
  generation:
    vllm_cfg:
      async_engine: true               # required: NeMo Gym uses async vLLM HTTP server
      expose_http_server: true         # required: exposes OpenAI-compatible /v1 endpoint
      enforce_eager: true              # disable Triton compilation to avoid NaN logprob bug

data:
  # MCQ + numeric subset (~45 000 train / ~468 val); generated with --question-type mcqa_and_numeric
  train_jsonl_fpath: ".../mmpr_tiny_train_mcqa_and_numeric.jsonl"
  validation_jsonl_fpath: ".../mmpr_tiny_val_mcqa_and_numeric.jsonl"

env:
  should_use_nemo_gym: true
  nemo_gym:
    config_paths:
      - responses_api_models/vllm_model/configs/vllm_model_for_training.yaml  # Required
      - resources_servers/mcqa/configs/mcqa.yaml                              # letter exact-match
      - resources_servers/math_with_judge/configs/math_with_judge.yaml        # math_verify; judge disabled below
    math_with_judge:
      resources_servers:
        math_with_judge:
          judge_model_server:
            name: policy_model   # required field; never called when should_use_judge is false
          should_use_judge: false  # math_verify library only — no LLM judge
    policy_model:
      responses_api_models:
        vllm_model:
          uses_reasoning_parser: true
          extra_body:
            chat_template_kwargs:
              enable_thinking: true    # enables <think> reasoning traces
```

> **Rule-based grading for MCQ + numeric.** Both `mcqa_simple_agent` (exact-match on `\boxed{X}`) and `math_with_judge_simple_agent` (SymPy-based symbolic equivalence via `math_verify`) are fully deterministic, low-latency, and require no extra GPU memory. To also cover the remaining ~29% free-form samples, swap `should_use_judge: false` → `true` and add an external OpenAI-compatible LLM endpoint via `resources_servers/equivalence_llm_judge/configs/equivalence_llm_judge.yaml`.

### Parallelism layout (1 node, 8× H100 80 GB)

| Component | Setting | Rationale |
|---|---|---|
| Megatron TP | 8 | All 8 GPUs in one tensor-parallel group (intra-NVLink) |
| Megatron EP | 8 | 128 routed experts → 16 per GPU (intra-node alltoall); DP = 8/(TP×PP) = 1 |
| Megatron PP | 1 | No pipeline parallel |
| vLLM TP | 8 | Full-node NVLink for fast generation |
| vLLM EP | 8 | Expert parallelism for vLLM inference |
| `gpu_memory_utilization` | 0.2 | Headroom shared with Megatron training weights |

You can copy the config to the NeMo RL root (all paths inside the config are container-internal) using the data preparation interactive job (or else, from the host environment with path adjustment):

In [ ]:
%%bash
# Copy config to NeMo RL examples directory (adjust paths as needed)
NEMORL=/shared/code/RL
COOKBOOK_DIR=/shared/code/Nemotron/usage-cookbook/Nemotron-3-Nano-Omni/grpo_nemo_gym

cp ${COOKBOOK_DIR}/grpo_mmpr_tiny_nanov3omni_gym.yaml \
   ${NEMORL}/examples/nemo_gym/grpo_mmpr_tiny_nanov3omni_gym.yaml

After this step, you can terminate the data preparation job with `scancel <SLURM-JOB-ID>`.

## Step 3. Launch training

### Option A: Interactive Ray cluster (recommended for first-time validation)

Spin up a Ray cluster without a pre-set command, then attach a shell to the head node and launch training manually. This lets you iterate quickly, inspect logs in real time, and catch errors before committing to a long batch run.

From your Slurm head node:

In [ ]:
%%bash
cd </YOUR/SHARED/NETWORK/STORAGE>/code/RL # Path from your host system

export ACCOUNT="YOUR_SLURM_ACCOUNT"
export PARTITION="YOUR_H100_PARTITION"
export CONTAINER="nvcr.io/<YOUR_NGC_ORG>/nemo-rl" # Supply the correct container image here
export MOUNTS="/lustre:/lustre,</YOUR/SHARED/NETWORK/STORAGE>:/shared"

export JOB_NAME="grpo-gym-interactive"
export NUM_NODES="1"
export GPUS_PER_NODE="8"
export TIME_LIMIT="04:00:00"

# No COMMAND — cluster stays idle, waiting for interactive submission
sbatch \
  --nodes="${NUM_NODES}" \
  --account="${ACCOUNT}" \
  --job-name="${JOB_NAME}" \
  --partition="${PARTITION}" \
  --time="${TIME_LIMIT}" \
  --gres="gpu:${GPUS_PER_NODE}" \
  --exclusive \
  --dependency=singleton \
  ray.sub

Upon successful submission, Slurm will print the job ID, for example:
```
Submitted batch job 1980204
```

Once the cluster is up, an attach script is created. Run it to get a shell on the head node:
```bash
bash 1980204-attach.sh
```

See https://github.com/NVIDIA-NeMo/RL/blob/main/docs/cluster.md for more details.

From the head node shell, launch training directly:

In [ ]:
%%bash
# --- Run from the Ray head node shell (inside the container) ---

export HF_HOME=/shared/HF_HOME
mkdir -p "${HF_HOME}/modules"  # Pre-create before workers start; avoids Lustre race on concurrent init_hf_modules() calls

export TORCH_EXTENSIONS_DIR=/tmp/torch_extensions
export TRITON_CACHE_DIR=/tmp/triton_cache
export PYTHONDONTWRITEBYTECODE=1  # Prevent concurrent .pyc write races when 8 Ray workers import the same packages simultaneously

export WANDB_API_KEY="YOUR_WANDB_KEY"

MODEL_PATH="/shared/models/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16"
TRAIN_JSONL="/shared/data/mmpr_tiny_gym/mmpr_tiny_train_mcqa_and_numeric.jsonl"
VAL_JSONL="/shared/data/mmpr_tiny_gym/mmpr_tiny_val_mcqa_and_numeric.jsonl"
CKPT_DIR="/shared/checkpoints/grpo-nanov3omni-mmpr-tiny-gym"
JOB_NAME="grpo-nanov3omni-mmpr-tiny-gym"

cd /opt/nemo-rl

NCCL_DEBUG=INFO \
NVTE_FWD_LAYERNORM_SM_MARGIN=16 \
NVTE_BWD_LAYERNORM_SM_MARGIN=16 \
NEMO_RL_LOG_GPU_MEMORY=1 \
CUDA_DEVICE_MAX_CONNECTIONS=1 \
NRL_IGNORE_VERSION_MISMATCH=true \
uv run examples/nemo_gym/run_grpo_nemo_gym.py \
    --config /shared/code/RL/examples/nemo_gym/grpo_mmpr_tiny_nanov3omni_gym.yaml \
    policy.model_name=${MODEL_PATH} \
    data.train_jsonl_fpath=${TRAIN_JSONL} \
    data.validation_jsonl_fpath=${VAL_JSONL} \
    checkpointing.checkpoint_dir=${CKPT_DIR} \
    logger.wandb_enabled=true \
    logger.wandb.name=${JOB_NAME}

### Option B: Slurm batch job (for production)

Once the interactive run confirms end-to-end correctness, submit as a fully automated batch job from the Slurm login node. The job starts a Ray cluster and runs training unattended.

In [ ]:
%%bash
NEMORL=</YOUR/SHARED/NETWORK/STORAGE>/code/RL # Note: this must be the host path
RAY_SUB="${NEMORL}/ray.sub"

# All paths below are as seen INSIDE the container
CONFIG_PATH="/shared/code/RL/examples/nemo_gym/grpo_mmpr_tiny_nanov3omni_gym.yaml"
MODEL_PATH="/shared/models/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16"
TRAIN_JSONL="/shared/data/mmpr_tiny_gym/mmpr_tiny_train_mcqa_and_numeric.jsonl"
VAL_JSONL="/shared/data/mmpr_tiny_gym/mmpr_tiny_val_mcqa_and_numeric.jsonl"
CKPT_DIR="/shared/checkpoints/grpo-nanov3omni-mmpr-tiny-gym"

export WANDB_API_KEY="YOUR_WANDB_KEY"
export ACCOUNT="YOUR_SLURM_ACCOUNT"
export PARTITION="YOUR_H100_PARTITION"
export CONTAINER="nvcr.io/<YOUR_NGC_ORG>/nemo-rl"
export MOUNTS="/lustre:/lustre,</YOUR/SHARED/NETWORK/STORAGE>:/shared"

export JOB_NAME="grpo-nanov3omni-mmpr-tiny-gym"
export NUM_NODES="1"
export GPUS_PER_NODE="8"
export TIME_LIMIT="04:00:00"

export HF_HOME="/shared/HF_HOME"
export NCCL_DEBUG=INFO
export NRL_IGNORE_VERSION_MISMATCH=true
export TORCH_EXTENSIONS_DIR=/tmp/torch_extensions
export TRITON_CACHE_DIR=/tmp/triton_cache
export PYTHONDONTWRITEBYTECODE=1  # Prevent concurrent .pyc write races when 8 Ray workers import the same packages simultaneously

# COMMAND must be set before sbatch so ray.sub picks it up
export COMMAND="\
mkdir -p ${HF_HOME}/modules && \\
cd /opt/nemo-rl && \\
NCCL_DEBUG=INFO \
NVTE_FWD_LAYERNORM_SM_MARGIN=16 \
NVTE_BWD_LAYERNORM_SM_MARGIN=16 \
NEMO_RL_LOG_GPU_MEMORY=1 \
CUDA_DEVICE_MAX_CONNECTIONS=1 \
NRL_IGNORE_VERSION_MISMATCH=true \
uv run examples/nemo_gym/run_grpo_nemo_gym.py \\
  --config ${CONFIG_PATH} \\
  policy.model_name=${MODEL_PATH} \\
  data.train_jsonl_fpath=${TRAIN_JSONL} \\
  data.validation_jsonl_fpath=${VAL_JSONL} \\
  checkpointing.checkpoint_dir=${CKPT_DIR} \\
  logger.wandb_enabled=true \\
  logger.wandb.name=${JOB_NAME}"

sbatch \
  --nodes="${NUM_NODES}" \
  --account="${ACCOUNT}" \
  --job-name="${JOB_NAME}" \
  --partition="${PARTITION}" \
  --time="${TIME_LIMIT}" \
  --gres="gpu:${GPUS_PER_NODE}" \
  --exclusive \
  --dependency=singleton \
  "${RAY_SUB}"

## Implementation details

NeMo Gym replaces the Nemo RL built-in `EnvironmentInterface` rollout loop with an OpenAI-compatible HTTP server, enabling flexible, pluggable verifiers (LLM-as-judge, rule-based MCQA, custom FastAPI servers) without modifying the core NeMo RL code.


### How NeMo Gym differs from the native environment path

| | Native env path | **NeMo Gym path (this notebook)** |
|---|---|---|
| Entry point | `examples/run_vlm_grpo.py` | `examples/nemo_gym/run_grpo_nemo_gym.py` |
| Rollout engine | vLLM sync engine | vLLM **async** engine + HTTP server |
| Reward / verifier | Python callable in `env:` config | NeMo Gym resource server (FastAPI) |
| Data format | HuggingFace dataset + processor | **JSONL** (`responses_create_params` schema) |
| Data prep step | Automatic on first run | Run `convert_mmpr_tiny_to_gym_jsonl.py` once |
| Multimodal dedup | `deduplicate_multimodal_data: true` | **Must be `false`** (async engine incompatible) |
| Verifier for MMPR | `verl_geo3k` rule scorer | `mcqa_simple_agent` (letter exact-match) + `math_with_judge_simple_agent` (`math_verify`, no LLM judge) — MCQ + numeric subset |